<a href="https://colab.research.google.com/github/SravanthiAluguri/FUTURE_ML_03/blob/main/Future_Intern_Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
data = {
    "candidate_name": ["Anu", "Rahul", "Priya", "Karan"],
    "resume_text": [
        "Python Machine Learning Deep Learning SQL Data Analysis",
        "Java Spring Boot MySQL HTML CSS",
        "Python Data Science NLP Machine Learning TensorFlow",
        "C++ Embedded Systems Microcontrollers"
    ]
}

df = pd.DataFrame(data)
job_description = "Looking for Python Machine Learning NLP SQL candidate with Data Science knowledge"
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df["clean_resume"] = df["resume_text"].apply(preprocess)
job_clean = preprocess(job_description)

vectorizer = TfidfVectorizer()

all_text = df["clean_resume"].tolist() + [job_clean]

tfidf_matrix = vectorizer.fit_transform(all_text)

resume_vectors = tfidf_matrix[:-1]
job_vector = tfidf_matrix[-1]
similarity_scores = cosine_similarity(resume_vectors, job_vector)

df["score"] = similarity_scores
ranked_df = df.sort_values(by="score", ascending=False)

print("Candidate Ranking:\n")
print(ranked_df[["candidate_name", "score"]])
# Extract keywords from job description
job_skills = set(job_clean.split())

def find_skill_gap(resume):
    resume_skills = set(resume.split())
    missing_skills = job_skills - resume_skills
    return missing_skills

ranked_df["missing_skills"] = ranked_df["clean_resume"].apply(find_skill_gap)

print("\nSkill Gap Analysis:\n")
print(ranked_df[["candidate_name", "missing_skills"]])

Candidate Ranking:

  candidate_name     score
2          Priya  0.517233
0            Anu  0.406573
1          Rahul  0.000000
3          Karan  0.000000

Skill Gap Analysis:

  candidate_name                                     missing_skills
2          Priya    {sql, with, candidate, looking, for, knowledge}
0            Anu  {with, candidate, nlp, science, looking, for, ...
1          Rahul  {sql, learning, python, with, candidate, nlp, ...
3          Karan  {sql, learning, python, with, candidate, nlp, ...


In [7]:
# ==========================================================
# RESUME / JOB ROLE MATCHING SYSTEM
# Dataset: Resume Entities & Job Roles Dataset (Kaggle)
# Uses ONLY attributes present in the dataset
# ==========================================================

import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

# ---------------- 1. Load Dataset ----------------
df = pd.read_csv("job_descriptions.csv")

# clean column names
df.columns = df.columns.str.lower().str.strip()

print("Columns:", df.columns)

# use correct attributes from dataset
df = df[['job title','job description','skills']]

df = df.dropna()

print("Dataset shape:", df.shape)

# ---------------- 2. Text Cleaning ----------------
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['cleaned_description'] = df['job description'].apply(clean_text)

# ---------------- 3. Convert Text to TF-IDF ----------------
vectorizer = TfidfVectorizer(max_features=5000)

tfidf_matrix = vectorizer.fit_transform(df['cleaned_description'])

# ---------------- 4. Choose Target Role From Dataset ----------------
# change this role if needed
target_role = "Data Scientist"

target_index = df[df['job title'] == target_role].index[0]

target_vector = tfidf_matrix[target_index]

# ---------------- 5. Compute Similarity ----------------
similarity_scores = cosine_similarity(tfidf_matrix, target_vector)

df['similarity_score'] = similarity_scores

# ---------------- 6. Rank Roles / Candidates ----------------
ranked_roles = df.sort_values(by='similarity_score', ascending=False)

print("\nTop matches for role:", target_role)
print(ranked_roles[['job title','similarity_score']].head(10))

# ---------------- 7. Skill Gap Identification ----------------
target_skills = set(df.loc[target_index,'skills'].lower().split(','))

def find_missing(candidate_skills):

    candidate_skills = set(str(candidate_skills).lower().split(','))

    missing = target_skills - candidate_skills

    return list(missing)

ranked_roles['missing_skills'] = ranked_roles['skills'].apply(find_missing)

print("\nSkill gaps for top matches:")
print(ranked_roles[['job title','missing_skills']].head(5))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Columns: Index(['job id', 'experience', 'qualifications', 'salary range', 'location',
       'country', 'latitude', 'longitude', 'work type', 'company size',
       'job posting date', 'preference', 'contact person', 'contact',
       'job title', 'role', 'job portal', 'job description', 'benefits',
       'skills', 'responsibilities', 'company', 'company profile'],
      dtype='object')
Dataset shape: (1615940, 3)

Top matches for role: Data Scientist
              job title  similarity_score
354420   Data Scientist               1.0
217343   Data Scientist               1.0
105386   Data Scientist               1.0
1010905  Data Scientist               1.0
1240458  Data Scientist               1.0
1541826  Data Scientist               1.0
1096018  Data Scientist               1.0
435796   Data Scientist               1.0
240540   Data Scientist               1.0
292618   Data Scientist               1.0

Skill gaps for top matches:
              job title missing_skills
354420   Data